In [2]:
import pandas as pd
import requests
from pathlib import Path
import duckdb

# Data Sources

Taxi Data: https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data via API
Census Tract Data: https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Census_Tracts/4hp8-2i8z/about_data 
Weather Data: https://mesonet.agron.iastate.edu/request/download.phtml?network=IL_ASOS
    https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=MDW&data=all&year1=2024&month1=1&day1=1&year2=2026&month2=5&day2=26&tz=America%2FChicago&format=onlycomma&latlon=yes&elev=yes&missing=null&trace=0.0001&direct=yes -> MDW

Chicago Landsmark Information (Points of interest): https://data.cityofchicago.org/Historic-Preservation/Individual-Landmarks-Map/bddq-yxar
Parks and Facilities: https://data.cityofchicago.org/Parks-Recreation/Parks-Facilities-Features-Shapefiles/thkh-m6bg/about_data - zip folder

Further possible
Bus and Rail Data: https://data.cityofchicago.org/Transportation/CTA-Ridership-Daily-Boarding-Totals/6iiy-9s97/about_data
Chicago Landsmark Information (Points of interest): https://data.cityofchicago.org/Historic-Preservation/Individual-Landmarks-Map/bddq-yxar


# Download Taxi Data via API
Load the data from "https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data" using SODA3 API and store it as CSV.

Pipeline:
SODA API → CSV as Raw Backup → DuckDB CSV → Parquet → Polars LazyFrame → scikit-learn

Data Prep and Cleaning Pipeline:
raw CSV
  ↓
bronze Parquet      # 1:1 aus CSV/API, möglichst unverändert
  ↓
silver Parquet      # gecleant, typisiert, gefiltert
  ↓
gold/features       # ML-Features, train/test-ready

Note: please add required packages to uv first:
```
uv add duckdb polars pyarrow 
```

In [ ]:
# API Parameter
DATASET_ID = "ajtu-isnz"
API_URL = f"https://data.cityofchicago.org/api/v3/views/ajtu-isnz/query.csv"
APP_TOKEN = "gD5sEc45CSmyjrr8F9ZJdao0Z"

output_path = Path("data/taxi_test_5k.csv")

# Query Parameter
LIMIT = 5_000

SELECT_FIELDS = []

WHERE_CLAUSE = ""

query = f"""
SELECT
    {", ".join(f"`{col}`" for col in SELECT_FIELDS)}
WHERE
    {WHERE_CLAUSE}
LIMIT {LIMIT}
"""

query = query if len(SELECT_FIELDS) > 0 and WHERE_CLAUSE != "" else f"""SELECT * LIMIT {LIMIT}"""

# Build payload
headers = {}

if APP_TOKEN and APP_TOKEN != "***":
    headers["X-App-Token"] = APP_TOKEN

payload = {
    "query": query
}

# CSV per Post streamen
with requests.post(
    API_URL,
    headers=headers,
    json=payload,
    stream=True,
    timeout=180
) as r:
    r.raise_for_status()

    with output_path.open("wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print(f"Download finished: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

Download finished: taxi_test_5k.csv
File size: 2.61 MB


# Create Parquet File for efficient Data Handling

## Taxi Data

In [ ]:
RAW_CSV = Path("raw_data/taxi_full_data.csv")
BRONZE_TAXI_PARQUET = Path("data/bronze_chicago_taxi_2024.parquet")

BRONZE_TAXI_PARQUET.parent.mkdir(parents=True, exist_ok=True)

duckdb.sql(f"""
COPY (
    SELECT *
    FROM read_csv_auto(
        '{RAW_CSV}',
        header = true,
        sample_size = -1
    )
)
TO '{BRONZE_TAXI_PARQUET}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

print(f"Parquet-Datei erstellt: {BRONZE_TAXI_PARQUET}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet-Datei erstellt: chicago_taxi_2024.parquet


## Weather Data

In [ ]:
RAW_CSV = Path("raw_data/MDW.csv")
BRONZE_WEATHER_PARQUET = Path("data/bronze_weatherdata.parquet")

BRONZE_WEATHER_PARQUET.parent.mkdir(parents=True, exist_ok=True)

duckdb.sql(f"""
COPY (
    SELECT *
    FROM read_csv_auto(
        '{RAW_CSV}',
        header = true,
        sample_size = -1
    )
)
TO '{BRONZE_WEATHER_PARQUET}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

print(f"Parquet-Datei erstellt: {BRONZE_WEATHER_PARQUET}")

Parquet-Datei erstellt: bronze_weatherdata.parquet


### Quick check parquet file

In [ ]:
duckdb.sql(f"""
SELECT count(*)
FROM read_parquet('{BRONZE_TAXI_PARQUET}')
""").show()

duckdb.sql(f"""
SELECT count(*)
FROM read_parquet('{BRONZE_WEATHER_PARQUET}')
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     15406960 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        24360 │
└──────────────┘

